In [1]:
import random

random.seed(0)


def random_functional_graph(n):
    return [random.randrange(n) for _ in range(n)]


def orbit_path(start, nxt, alive):
    seen = {}
    path = []
    cur = start
    while alive[cur] and cur not in seen:
        seen[cur] = len(path)
        path.append(cur)
        cur = nxt[cur]
    return path


# seems like this really is the best sampling scheme (althrough the specific implementation could be optimized) the only thing to consider is balance of path lengths
# TODO: this could be proven to be optimal by taking a theoretical optimal bound and showing that this scheme achieves it
#   - althrough i cannot find a fine closed form for the plot in terms of s, n
def greedy_longest_paths(nxt):
    n = len(nxt)
    alive = [True] * n
    remaining = n
    path_lengths = []

    while remaining > 0:
        best_path = None
        best_len = -1
        for s in range(n):
            if not alive[s]:
                continue
            p = orbit_path(s, nxt, alive)
            if len(p) > best_len:
                best_len = len(p)
                best_path = p

        for v in best_path:
            if alive[v]:
                alive[v] = False
                remaining -= 1

        path_lengths.append(len(best_path))
    return path_lengths


def f_from_lengths(path_lengths, n):
    vals = [0] * (n + 1)
    total = 0
    prev = 1
    for i, L in enumerate(path_lengths, start=1):
        total += L
        upto = min(total, n)
        for s in range(prev, upto + 1):
            vals[s] = i
        prev = upto + 1
        if prev > n:
            break
    return vals


def expected_ratio(n=1000, trials=100):
    acc = [0.0] * (n + 1)
    for _ in range(trials):
        nxt = random_functional_graph(n)
        lengths = greedy_longest_paths(nxt)
        vals = f_from_lengths(lengths, n)
        for i in range(1, n + 1):
            acc[i] += vals[i] / i
    return [x / trials for x in acc]


# plot with varing n and plotly
import plotly.graph_objects as go


def plot_expected_ratios(n_values, trials=100):
    fig = go.Figure()
    for n in n_values:
        ratio = expected_ratio(n, trials)
        fig.add_trace(
            go.Scatter(
                x=[i / n for i in range(1, n + 1)],
                y=ratio[1:],
                mode="lines",
                name=f"n={n}",
            )
        )
    fig.update_layout(
        title="Normalized Expected Paths for Varying n",
        xaxis_title="samples",
        yaxis_title="E[f(samples)] / samples",
        legend_title="n",
    )
    fig.show()


n_values = [32, 64, 128, 256]
plot_expected_ratios(n_values, trials=100)